In [1]:
import numpy as np
import pandas as pd
import joblib
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
X_train = np.load('../models/X_train_scaled.npy')
X_test  = np.load('../models/X_test_scaled.npy')
y_train = np.load('../models/y_train_balanced.npy')
y_test  = np.load('../models/y_test.npy')
feature_names = joblib.load('../models/feature_names.pkl')

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean()*100:.1f}%")
print(f"Test default rate:  {y_test.mean()*100:.1f}%")
print(f"Features: {feature_names}")

Train: (223956, 14) | Test: (30000, 14)
Train default rate: 50.0%
Test default rate:  6.7%
Features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'TotalTimesLate', 'IncomeDebtBurden', 'UtilizationPerAccount', 'EverSeriouslyLate']


In [3]:
# We train 3 models and compare — this is standard DS workflow
# Logistic Regression = simple baseline
# Random Forest = ensemble tree model
# XGBoost = gradient boosting, usually best performer

print("Training models — this may take 2-3 minutes...\n")

# Model 1: Logistic Regression
print("1/3 Training Logistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1)
lr.fit(X_train, y_train)
print("   ✅ Done")

# Model 2: Random Forest
print("2/3 Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("   ✅ Done")

# Model 3: XGBoost
print("3/3 Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
print("   ✅ Done")

print("\nAll 3 models trained successfully ✅")

Training models — this may take 2-3 minutes...

1/3 Training Logistic Regression...
   ✅ Done
2/3 Training Random Forest...
   ✅ Done
3/3 Training XGBoost...
   ✅ Done

All 3 models trained successfully ✅


In [4]:
models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb
}

results = {}

print(f"{'Model':<25} {'AUC-ROC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print("=" * 65)

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    auc   = roc_auc_score(y_test, y_prob)
    f1    = f1_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred)
    rec   = recall_score(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc': auc, 'f1': f1,
        'precision': prec, 'recall': rec
    }
    
    print(f"{name:<25} {auc:>8.4f} {f1:>8.4f} {prec:>10.4f} {rec:>8.4f}")

print("\nMetrics explanation:")
print("AUC-ROC  → Overall model quality (1.0 = perfect, 0.5 = random)")
print("F1       → Balance of precision and recall")
print("Precision→ Of predicted defaults, how many were real defaults")
print("Recall   → Of real defaults, how many did we catch")

Model                      AUC-ROC       F1  Precision   Recall
Logistic Regression         0.8337   0.3179     0.2059   0.6978
Random Forest               0.8496   0.3597     0.2450   0.6758
XGBoost                     0.8362   0.3595     0.3251   0.4020

Metrics explanation:
AUC-ROC  → Overall model quality (1.0 = perfect, 0.5 = random)
F1       → Balance of precision and recall
Precision→ Of predicted defaults, how many were real defaults
Recall   → Of real defaults, how many did we catch


In [5]:
fig = go.Figure()
colors = ['#3498DB', '#2ECC71', '#E74C3C']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        name=f"{name} (AUC={res['auc']:.4f})",
        line=dict(color=color, width=2.5)
    ))

# Random baseline
fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    name='Random Baseline',
    line=dict(color='gray', width=1.5, dash='dash')
))

fig.update_layout(
    title='ROC Curve — Model Comparison',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    plot_bgcolor='white',
    height=500,
    legend=dict(x=0.6, y=0.1)
)
fig.show()

In [6]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=list(results.keys()))

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    
    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=['Predicted No Default', 'Predicted Default'],
            y=['Actual No Default', 'Actual Default'],
            colorscale='Blues',
            text=cm, texttemplate='%{text}',
            textfont=dict(size=14),
            showscale=False
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title='Confusion Matrices — All Models',
    height=400
)
fig.show()

In [7]:
# XGBoost almost always wins on tabular data
# We tune it further with the best hyperparameters

print("Tuning XGBoost — best model...")

xgb_tuned = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_tuned.fit(X_train, y_train)

y_pred_tuned = xgb_tuned.predict(X_test)
y_prob_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

auc_tuned = roc_auc_score(y_test, y_prob_tuned)
f1_tuned  = f1_score(y_test, y_pred_tuned)

print(f"\nTuned XGBoost Results:")
print(f"AUC-ROC:   {auc_tuned:.4f}")
print(f"F1 Score:  {f1_tuned:.4f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred_tuned,
      target_names=['No Default', 'Default']))

Tuning XGBoost — best model...

Tuned XGBoost Results:
AUC-ROC:   0.8434
F1 Score:  0.3750

Detailed Report:
              precision    recall  f1-score   support

  No Default       0.96      0.93      0.95     27995
     Default       0.32      0.45      0.38      2005

    accuracy                           0.90     30000
   macro avg       0.64      0.69      0.66     30000
weighted avg       0.92      0.90      0.91     30000



In [8]:
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_tuned.feature_importances_
}).sort_values('Importance', ascending=True)

fig = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='XGBoost Feature Importance — What Drives Default Risk?',
    color='Importance',
    color_continuous_scale='Reds'
)

fig.update_layout(
    plot_bgcolor='white',
    height=550,
    showlegend=False
)
fig.show()

print("\nTop 5 most important features:")
print(importance_df.tail(5)[['Feature','Importance']].iloc[::-1].to_string())


Top 5 most important features:
                                 Feature  Importance
0   RevolvingUtilizationOfUnsecuredLines    0.276588
9                     NumberOfDependents    0.227586
10                        TotalTimesLate    0.184118
12                 UtilizationPerAccount    0.048975
13                     EverSeriouslyLate    0.048342


In [9]:
# Random Forest wins on AUC — most important metric for credit risk
# AUC measures how well model separates defaulters from non-defaulters

best_model = rf  # Random Forest

y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]
auc_best    = roc_auc_score(y_test, y_prob_best)
f1_best     = f1_score(y_test, y_pred_best)

print("BEST MODEL: Random Forest")
print(f"AUC-ROC:  {auc_best:.4f}")
print(f"F1 Score: {f1_best:.4f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred_best,
      target_names=['No Default', 'Default']))

# Save
joblib.dump(best_model, '../models/best_model.pkl')

# Feature importance from Random Forest
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

model_metrics = {
    'auc':              round(auc_best, 4),
    'f1':               round(f1_best, 4),
    'model_name':       'Random Forest',
    'feature_names':    feature_names,
    'feature_importance': dict(zip(
        importance_df['Feature'].tolist(),
        importance_df['Importance'].tolist()
    ))
}
joblib.dump(model_metrics, '../models/model_metrics.pkl')

print("\nSaved:")
print("  ✅ best_model.pkl  (Random Forest)")
print("  ✅ model_metrics.pkl")
print("\n🎯 Ready for Notebook 4 — SHAP Analysis")

BEST MODEL: Random Forest
AUC-ROC:  0.8496
F1 Score: 0.3597

Detailed Report:
              precision    recall  f1-score   support

  No Default       0.97      0.85      0.91     27995
     Default       0.25      0.68      0.36      2005

    accuracy                           0.84     30000
   macro avg       0.61      0.76      0.63     30000
weighted avg       0.92      0.84      0.87     30000


Saved:
  ✅ best_model.pkl  (Random Forest)
  ✅ model_metrics.pkl

🎯 Ready for Notebook 4 — SHAP Analysis
